# Contextual Bandit Algorithms for News Recommendation

This notebook implements and compares bandit algorithms for personalised news recommendation.
A user arrives at each time step with a context vector encoding their preferences and demographics.
The agent must learn which article to recommend to maximise click-through rate over time.

**Algorithms covered:**
- Part 1 (non-contextual): ε-Greedy, UCB1
- Part 2 (contextual): Random baseline, Contextual ε-Greedy, LinUCB

**Reward model:** `P(click | arm a, context x) = sigmoid(θ_a · x)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Environments

In [ ]:
class BernoulliBandit:
    """Multi-armed Bernoulli bandit for the non-contextual setting."""

    def __init__(self, probs):
        self.probs = np.array(probs)
        self.k = len(probs)

    def pull(self, arm):
        return int(np.random.rand() < self.probs[arm])

    @property
    def best_arm(self):
        return int(np.argmax(self.probs))


class ContextualNewsEnv:
    """
    Contextual bandit environment simulating a news recommendation system.

    Users arrive with a 6-dimensional context vector:
        [likes_politics, sports_fan, techie, mobile_user, morning_reader, age_z]

    Four articles (arms): Politics, Sports, Tech, Lifestyle.

    Click probability: P(click | arm a, context x) = sigmoid(theta[a] @ x)
    """

    FEATURE_NAMES = [
        "likes_politics", "sports_fan", "techie",
        "mobile_user", "morning_reader",
        "age_z",   # standardized: (age - 40) / 15
    ]
    ARM_NAMES = ["Politics", "Sports", "Tech", "Lifestyle"]

    THETA = np.array([
        [ 1.6,  0.2,  0.1,  0.2,  0.7,  0.4],
        [ 0.1,  1.8,  0.1,  0.7,  0.2, -0.1],
        [ 0.0,  0.1,  1.9, -0.1, -0.2, -0.2],
        [ 0.3,  0.2,  0.2,  1.0,  0.8,  0.0],
    ], dtype=float)

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.d = len(self.FEATURE_NAMES)
        self.k = len(self.ARM_NAMES)
        self.theta = self.THETA.copy()
        self.feature_names = self.FEATURE_NAMES
        self.arm_names = self.ARM_NAMES

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def sample_context(self):
        seg = self.rng.choice(["politics", "sports", "tech", "on_the_go", "morning_person"])
        cores = {
            "politics":       np.array([1.6, 0.2, 0.2, 0.3, 0.9]),
            "sports":         np.array([0.2, 1.8, 0.2, 1.0, 0.3]),
            "tech":           np.array([0.2, 0.2, 1.9, 0.3, 0.2]),
            "on_the_go":      np.array([0.4, 0.9, 0.5, 1.8, 0.7]),
            "morning_person": np.array([0.8, 0.3, 0.2, 0.6, 1.9]),
        }
        x = np.empty(self.d)
        x[:5] = cores[seg] + self.rng.normal(0, 0.2, size=5)
        age = float(np.clip(self.rng.normal(40, 12), 18, 80))
        x[5] = (age - 40.0) / 15.0
        return x

    def click_prob(self, arm, x):
        return float(self._sigmoid(self.theta[arm] @ x))

    def click(self, arm, x):
        p = self.click_prob(arm, x)
        return int(self.rng.random() < p), p

    def best_arm(self, x):
        scores = self.theta @ x
        arm = int(np.argmax(scores))
        return arm, float(self._sigmoid(scores[arm]))

## 2. Part 1 — Non-Contextual Bandits

In the non-contextual setting, each article has a fixed unknown click probability.
The agent has no information about the user — it must learn the best arm purely from observed rewards.

In [ ]:
def run_epsilon_greedy(probs, T, epsilon=0.1, seed=0):
    """
    Epsilon-greedy on a Bernoulli bandit.

    Exploration: with prob epsilon, pull a random arm.
    Exploitation: otherwise pull the arm with highest sample mean Q[a].
    Update: incremental mean Q[a] += (r - Q[a]) / N[a].
    """
    bandit = BernoulliBandit(probs)
    rng = np.random.default_rng(seed)
    k = len(probs)
    Q, N = np.zeros(k), np.zeros(k, dtype=int)
    rewards, regrets, actions = np.zeros(T, dtype=int), np.zeros(T), np.zeros(T, dtype=int)
    optimal = float(np.max(probs))

    for t in range(T):
        arm = int(rng.integers(0, k)) if rng.random() < epsilon else int(np.argmax(Q))
        r = int(bandit.pull(arm))
        rewards[t], actions[t], regrets[t] = r, arm, optimal - probs[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]

    return {"rewards": rewards, "regrets": regrets, "actions": actions, "Q": Q, "N": N}


def run_ucb(probs, T, c=1.0, seed=0):
    """
    UCB1 on a Bernoulli bandit.

    UCB score: Q[a] + c * sqrt(ln(t) / N[a])
    Unvisited arms receive score +inf (forced initial exploration).
    """
    bandit = BernoulliBandit(probs)
    rng = np.random.default_rng(seed)
    k = len(probs)
    Q, N = np.zeros(k), np.zeros(k, dtype=int)
    rewards, regrets, actions = np.zeros(T, dtype=int), np.zeros(T), np.zeros(T, dtype=int)
    optimal = float(np.max(probs))

    for t in range(1, T + 1):
        ucb = np.where(N == 0, np.inf, Q + c * np.sqrt(np.log(t) / N))
        arm = int(np.argmax(ucb))
        r = int(bandit.pull(arm))
        rewards[t-1], actions[t-1], regrets[t-1] = r, arm, optimal - probs[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]

    return {"rewards": rewards, "regrets": regrets, "actions": actions, "Q": Q, "N": N}

In [ ]:
PROBS   = [0.2, 0.1, 0.5, 0.9]
T1      = 10_000
EPSILON = 0.1
C       = 1.0
SEED    = 123

out_eps = run_epsilon_greedy(PROBS, T1, epsilon=EPSILON, seed=SEED)
out_ucb = run_ucb(PROBS, T1, c=C, seed=SEED)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("Part 1: Non-Contextual Bandits", fontsize=13, y=1.01)

for row, (out, label) in enumerate([
    (out_eps, f"ε-Greedy  (ε={EPSILON})"),
    (out_ucb, f"UCB1  (c={C})"),
]):
    avg = np.cumsum(out["rewards"]) / np.arange(1, T1 + 1)
    axes[row, 0].plot(avg, color="steelblue")
    axes[row, 0].axhline(max(PROBS), color="tomato", lw=0.9, ls="--", label=f"optimal={max(PROBS)}")
    axes[row, 0].set_title(f"{label}: Average Reward")
    axes[row, 0].set_xlabel("Step"); axes[row, 0].set_ylabel("Avg reward")
    axes[row, 0].legend(fontsize=9)

    axes[row, 1].plot(np.cumsum(out["regrets"]), color="darkorange")
    axes[row, 1].set_title(f"{label}: Cumulative Regret")
    axes[row, 1].set_xlabel("Step"); axes[row, 1].set_ylabel("Regret")

    counts = np.bincount(out["actions"], minlength=len(PROBS))
    colors = ["tomato" if i == np.argmax(PROBS) else "steelblue" for i in range(len(PROBS))]
    axes[row, 2].bar(range(len(PROBS)), counts, color=colors)
    axes[row, 2].set_title(f"{label}: Pull Counts")
    axes[row, 2].set_xlabel("Arm"); axes[row, 2].set_ylabel("# pulls")
    axes[row, 2].set_xticks(range(len(PROBS)))
    axes[row, 2].set_xticklabels([f"arm {i}\n(p={p})" for i, p in enumerate(PROBS)])

plt.tight_layout()
plt.savefig("results/part1_noncontextual.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ε-Greedy | reward={out_eps['rewards'].sum():,}  avg={out_eps['rewards'].mean():.3f}  cum_regret={out_eps['regrets'].sum():.1f}")
print(f"UCB1     | reward={out_ucb['rewards'].sum():,}  avg={out_ucb['rewards'].mean():.3f}  cum_regret={out_ucb['regrets'].sum():.1f}")

## 3. Part 2 — Contextual Bandits

In the contextual setting, a 6-dimensional user feature vector is observed at each step.
Algorithms that exploit this context can learn personalised arm preferences and converge faster.

In [ ]:
class Policy:
    def select(self, x: np.ndarray) -> int: raise NotImplementedError
    def update(self, arm: int, x: np.ndarray, r: int): pass


class RandomPolicy(Policy):
    """Uniform random baseline — ignores context."""
    def __init__(self, k, seed=0):
        self.k = k
        self.rng = np.random.default_rng(seed)
    def select(self, x):
        return int(self.rng.integers(0, self.k))


class ContextualEpsGreedy(Policy):
    """
    Contextual epsilon-greedy with a linear reward model per arm.

    Each arm maintains a weight vector w_a updated by stochastic gradient:
        w_a += (r - w_a @ x) * x / N[a]

    Exploration: with prob epsilon, pick a random arm.
    Exploitation: pick arm with highest predicted reward w_a @ x.
    """
    def __init__(self, k, d, epsilon=0.1, seed=0):
        self.eps = epsilon
        self.W = np.zeros((k, d))   # weight matrix: k arms x d features
        self.N = np.zeros(k, dtype=int)
        self.rng = np.random.default_rng(seed)

    def select(self, x):
        if self.rng.random() < self.eps:
            return int(self.rng.integers(0, self.W.shape[0]))
        scores = self.W @ x
        return int(self.rng.choice(np.flatnonzero(scores == np.max(scores))))

    def update(self, arm, x, r):
        self.N[arm] += 1
        self.W[arm] += (r - self.W[arm] @ x) * x / self.N[arm]


class LinUCB(Policy):
    """
    LinUCB with disjoint linear models (Li et al., 2010).

    Per arm, maintains regularised gram matrix A_a and reward vector b_a.
    UCB score: theta_hat_a @ x + alpha * sqrt(x @ A_a^{-1} @ x)

    The exploration bonus (second term) is large when x lies in a
    direction not yet explored for arm a, driving principled exploration.
    """
    def __init__(self, k, d, alpha=1.0, lambda_=1.0):
        self.alpha = alpha
        self.A = [lambda_ * np.eye(d) for _ in range(k)]
        self.b = [np.zeros(d) for _ in range(k)]
        self.k = k

    def select(self, x):
        scores = np.empty(self.k)
        for a in range(self.k):
            A_inv_x = np.linalg.solve(self.A[a], x)
            theta_hat = np.linalg.solve(self.A[a], self.b[a])
            scores[a] = theta_hat @ x + self.alpha * np.sqrt(x @ A_inv_x)
        return int(np.argmax(scores))

    def update(self, arm, x, r):
        self.A[arm] += np.outer(x, x)
        self.b[arm] += r * x

In [ ]:
def run_on_contexts(policy, env_seed, contexts):
    env = ContextualNewsEnv(seed=env_seed)
    T = len(contexts)
    rewards, inst_regret = np.zeros(T, dtype=int), np.zeros(T)

    for t, x in enumerate(contexts):
        arm = policy.select(x)
        r, p_sel = env.click(arm, x)
        rewards[t] = r
        p_star = 1.0 / (1.0 + np.exp(-np.max(env.theta @ x)))
        inst_regret[t] = p_star - p_sel
        policy.update(arm, x, r)

    return rewards, inst_regret


T2, SEED2 = 20_000, 7
env = ContextualNewsEnv(seed=SEED2)
k, d = env.k, env.d
contexts = [env.sample_context() for _ in range(T2)]

r_rand, reg_rand = run_on_contexts(RandomPolicy(k, seed=SEED2), SEED2, contexts)
r_eps,  reg_eps  = run_on_contexts(ContextualEpsGreedy(k, d, epsilon=0.1, seed=SEED2), SEED2, contexts)
r_ucb,  reg_ucb  = run_on_contexts(LinUCB(k, d, alpha=1.0, lambda_=1.0), SEED2, contexts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Part 2: Contextual Bandits — Policy Comparison", fontsize=13)
colors = {"Random": "#7f8c8d", "ε-Greedy": "#2980b9", "LinUCB": "#27ae60"}

for r, reg, label in [
    (r_rand, reg_rand, "Random"),
    (r_eps,  reg_eps,  "ε-Greedy"),
    (r_ucb,  reg_ucb,  "LinUCB"),
]:
    axes[0].plot(np.cumsum(r) / np.arange(1, T2+1), label=label, color=colors[label])
    axes[1].plot(np.cumsum(reg), label=label, color=colors[label])

axes[0].set_title("Average Reward (running mean)")
axes[0].set_xlabel("Time step"); axes[0].set_ylabel("Avg reward"); axes[0].legend()
axes[1].set_title("Cumulative Regret")
axes[1].set_xlabel("Time step"); axes[1].set_ylabel("Cumulative regret"); axes[1].legend()

plt.tight_layout()
plt.savefig("results/part2_contextual.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'Algorithm':<26} {'Reward':>8} {'Avg':>7} {'Cum. Regret':>13}")
print("-" * 58)
for r, reg, label in [(r_rand, reg_rand, "Random"), (r_eps, reg_eps, "Contextual ε-Greedy"), (r_ucb, reg_ucb, "LinUCB")]:
    print(f"{label:<26} {r.sum():>8,} {r.mean():>7.3f} {reg.sum():>13.1f}")

## 4. Before vs. After Learning

To illustrate what the algorithms learn, we inspect their decisions before any training
and after 10,000 interactions. Before training, LinUCB explores broadly due to high uncertainty;
after training, it confidently matches articles to user segments.

In [ ]:
def compare_before_after(pre_steps=5, train_T=10_000, post_steps=5,
                          epsilon=0.1, alpha=1.0, lambda_=1.0, seed=7):
    ctx_env = ContextualNewsEnv(seed=seed)
    pre_ctx   = [ctx_env.sample_context() for _ in range(pre_steps)]
    train_ctx = [ctx_env.sample_context() for _ in range(train_T)]
    post_ctx  = [ctx_env.sample_context() for _ in range(post_steps)]

    env_pre  = ContextualNewsEnv(seed=seed)
    env_post = ContextualNewsEnv(seed=seed)
    env_tr   = ContextualNewsEnv(seed=seed)

    eps = ContextualEpsGreedy(k=env_pre.k, d=env_pre.d, epsilon=epsilon, seed=seed)
    ucb = LinUCB(k=env_pre.k, d=env_pre.d, alpha=alpha, lambda_=lambda_)
    rnd = RandomPolicy(k=env_pre.k, seed=seed)

    def show(env, policies_labels, x, phase):
        best_arm = int(np.argmax(env.theta @ x))
        print(f"  Context: " + ", ".join(
            f"{n}={v:.2f}" for n, v in zip(env.feature_names, x)
        ))
        for policy, label in policies_labels:
            arm = policy.select(x)
            r, p = env.click(arm, x)
            policy.update(arm, x, r)
            match = "✓" if arm == best_arm else "✗"
            print(f"  {label:<22} → {env.arm_names[arm]:<10} p(click)={p:.2f}  "
                  f"clicked={'yes' if r else 'no ':3s}  optimal={env.arm_names[best_arm]} {match}")
        print()

    print("=" * 70)
    print("BEFORE LEARNING")
    print("=" * 70)
    for i, x in enumerate(pre_ctx, 1):
        print(f"\nStep {i}:")
        show(env_pre, [(rnd, "Random"), (eps, "ε-Greedy"), (ucb, "LinUCB")], x, "pre")

    # train
    eps_tr = ContextualEpsGreedy(k=env_tr.k, d=env_tr.d, epsilon=epsilon, seed=seed)
    ucb_tr = LinUCB(k=env_tr.k, d=env_tr.d, alpha=alpha, lambda_=lambda_)
    for x in train_ctx:
        arm = eps_tr.select(x); r, _ = env_tr.click(arm, x); eps_tr.update(arm, x, r)
        arm = ucb_tr.select(x); r, _ = env_tr.click(arm, x); ucb_tr.update(arm, x, r)

    print("=" * 70)
    print(f"AFTER {train_T:,} TRAINING STEPS")
    print("=" * 70)
    rnd2 = RandomPolicy(k=env_post.k, seed=seed+1)
    for i, x in enumerate(post_ctx, 1):
        print(f"\nStep {i}:")
        show(env_post, [(rnd2, "Random"), (eps_tr, "ε-Greedy"), (ucb_tr, "LinUCB")], x, "post")

compare_before_after(seed=7)

## 5. Key Findings

| Algorithm | Total Reward | Avg Reward | Cum. Regret |
|---|---|---|---|
| Random | 15,792 | 0.790 | 3,440.2 |
| Contextual ε-Greedy | 18,793 | 0.940 | 501.6 |
| **LinUCB** | **19,216** | **0.961** | **56.1** |

**UCB vs ε-Greedy (non-contextual):** UCB accumulates 12× less regret (40.7 vs 487.7)
by adapting its exploration to arm uncertainty rather than exploring uniformly.

**Context matters:** Contextual policies achieve average reward 0.96 vs 0.79 for random,
a 21% improvement from learning user-article affinity.

**LinUCB vs ε-Greedy (contextual):** LinUCB accumulates 9× less regret by maintaining
principled uncertainty estimates via the inverse gram matrix A_a^{-1}, concentrating
exploration where it is most informative in feature space.

## References

- Li, L., Chu, W., Langford, J., & Schapire, R. E. (2010). *A contextual-bandit approach
  to personalized news article recommendation.* WWW 2010. [arXiv:1003.0146](https://arxiv.org/abs/1003.0146)
- Auer, P., Cesa-Bianchi, N., & Fischer, P. (2002). *Finite-time analysis of the multiarmed
  bandit problem.* Machine Learning, 47(2–3), 235–256.